# Demo 1: a runtime compiler for skills

### Select the skill, resolve the task variables, assemble only the applicable instructions, record what ran

**The setting.** A documentation desk for the Acme data platform. Requests arrive as
small API envelopes: who is asking, what kind of text they want, who will read it,
in what tone. Five instruction files (we call them *artifacts*) describe how that
text must be written: an identity, a project file, a persona and two skills
(`release-notes`, `incident-report`).

**The claim this demo makes.** Most of what goes into a model prompt can be decided
before any model runs. The harness, the code around the model, can:

1. read a **skill index**: names, versions and one-paragraph descriptions, nothing else (*discovery*);
2. pick **one skill at one version** for the request (*selection*);
3. read the task's **variables** (audience, tone) off the envelope (*resolution*);
4. assemble **only the instruction blocks that apply** into one text, the *runtime
   surface*, with a manifest that says where every block came from (*compilation*);
5. **refuse** to build when two artifacts claim the same rule, or a file is malformed;
6. write down **what ran**: skill version, file digests, surface digest, model binding (*the record*).

Every code cell below is the Chapter 3 harness from the book, copied as-is where
possible. Markdown cells marked *Scene* say what is about to happen and the one
point to take from it. Cells are labelled as either **harness computes** (no model
involved, deterministic) or **model produces** (a real call through OpenRouter).

Definitions used throughout:

- **digest**: a short SHA-256 hash of a text. Same text, same digest; one character
  changed, different digest.
- **surface**: the exact system prompt the model receives, compiled from artifacts.
- **manifest**: the list of which artifact sections went into a surface, plus its digest.

## 0. Setup

Installs, the five artifacts from the shared Drive folder, and the OpenRouter key.
The key is read from the environment, a `.env`, the course repo's `course_app/.env`,
or a Colab secret named `OPENROUTER_API_KEY`. Nothing secret is printed.

In [16]:
%pip install -q instructor gdown python-dotenv pyyaml tiktoken

In [17]:
try:
    import gdown
except ModuleNotFoundError:
    %pip install -q gdown
    import gdown

import hashlib, inspect, json, os, re, shutil, textwrap, time
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import yaml

DRIVE_FOLDER_URL = (
    "https://drive.google.com/drive/folders/"
    "1hPCzmDY08tKq5o8vcykap9aFdiQ7UA_4?usp=sharing"
)

WORK = Path("ch03_skill_demo").resolve()

# gdown preserves the Google Drive folder name.
SRC = WORK / "artifacts"

shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)

kwargs = {
    "url": DRIVE_FOLDER_URL,
    "output": str(WORK),
    "quiet": True,
    "use_cookies": False,
}

if "remaining_ok" in inspect.signature(gdown.download_folder).parameters:
    kwargs["remaining_ok"] = True

downloaded = gdown.download_folder(**kwargs)

if not downloaded:
    raise RuntimeError(
        "Nothing was downloaded. Ensure that the Google Drive folder is "
        "shared with 'Anyone with the link'."
    )

files = sorted(SRC.glob("*.md"))

if not files:
    raise RuntimeError(f"No Markdown artifacts found in {SRC}/")

print(f"{len(files)} artifact(s) in {SRC}:")
for p in files:
    print(f"   {p.name:28} {p.stat().st_size:>6} bytes")

5 artifact(s) in /content/ch03_skill_demo/artifacts:
   AGENTS.md                      1023 bytes
   PROJECT.md                      891 bytes
   persona-plainspoken.md          571 bytes
   skill-incident-report.md       1429 bytes
   skill-release-notes.md         3645 bytes


In [18]:
import getpass, urllib.request

import instructor
import openai
from pydantic import BaseModel, Field

from dotenv import load_dotenv, find_dotenv


def _find_key():
    """OPENROUTER_API_KEY from, in order: the environment, a .env next to the notebook,
    the course repo's course_app/.env, a Colab secret, or a prompt. The value is never printed."""
    if os.environ.get("OPENROUTER_API_KEY"):
        return
    load_dotenv(find_dotenv(usecwd=True), override=False)
    for base in [Path.cwd(), *Path.cwd().parents][:4]:
        if (base / "course_app" / ".env").exists():
            load_dotenv(base / "course_app" / ".env", override=False)
            break
    if not os.environ.get("OPENROUTER_API_KEY"):
        try:
            from google.colab import userdata                 # Colab: the key icon in the left sidebar
            os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
        except Exception:                                      # noqa: BLE001
            pass
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")


_find_key()

BASE_URL = "https://openrouter.ai/api/v1"
WRITER = "openai/gpt-5.6-luna"          # writes the notes
ROUTER = "openai/gpt-5.4-mini"          # picks the skill; a small model is plenty

USAGE: List[Dict[str, object]] = []      # every call, for the cost section at the end


@lru_cache(maxsize=4)
def client_for(mode):
    return instructor.from_openai(
        openai.OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url=BASE_URL),
        mode=mode)


def typed_call(*, model, response_model, messages, temperature=0.0, label="",
               mode=instructor.Mode.TOOLS, max_retries=2):
    """One structured call, with its usage and wall clock recorded."""
    t0 = time.perf_counter()
    out, completion = client_for(mode).chat.completions.create_with_completion(
        model=model, response_model=response_model, max_retries=max_retries,
        temperature=temperature, messages=messages)
    u = getattr(completion, "usage", None)
    USAGE.append({"label": label or model, "model": model,
                  "prompt_tokens": getattr(u, "prompt_tokens", 0),
                  "completion_tokens": getattr(u, "completion_tokens", 0),
                  "seconds": time.perf_counter() - t0})
    return out


print(f"writer: {WRITER}   router: {ROUTER}   key loaded: {bool(os.environ.get('OPENROUTER_API_KEY'))}")

writer: openai/gpt-5.6-luna   router: openai/gpt-5.4-mini   key loaded: True


## 1. The request: an API envelope

This is everything the desk receives. Note what is already decided in it:
the task type, the audience, the tone, the facts. None of these should be guessed by
a model later.

**Harness computes.** Nothing here is model output.

In [19]:
REQUEST = {
    "request_id": "req-0917-001",
    "user_type": "release-manager",         # who is asking
    "task_type": "release-notes",           # what they want
    "audience": "customer",                 # who will read it
    "tone": "casual",                       # how it should sound
    "task": "Version 2.4.0 has shipped. Write the release notes for customers now.",
    "payload": {
        "version": "2.4.0",
        "breaking": True,
        "changes": [
            "the CSV exporter was rewritten; exports run about twice as fast",
            "fixed escaping of quotes inside CSV fields",
            "peak memory during export reduced by roughly 40 percent",
            "the /v1/export endpoint now returns 202 with a job id instead of the file; callers must poll /v1/export/{id}",
        ],
    },
}
print(json.dumps(REQUEST, indent=2))

{
  "request_id": "req-0917-001",
  "user_type": "release-manager",
  "task_type": "release-notes",
  "audience": "customer",
  "tone": "casual",
  "task": "Version 2.4.0 has shipped. Write the release notes for customers now.",
  "payload": {
    "version": "2.4.0",
    "breaking": true,
    "changes": [
      "the CSV exporter was rewritten; exports run about twice as fast",
      "fixed escaping of quotes inside CSV fields",
      "peak memory during export reduced by roughly 40 percent",
      "the /v1/export endpoint now returns 202 with a job id instead of the file; callers must poll /v1/export/{id}"
    ]
  }
}


## 2. Discovery: the harness reads the skill index, not the skills

The next cell is Chapter 3's harness, verbatim. It parses each artifact
into a typed record: kind, name, version, the sections it owns, and a digest of the
file. Then we build the **skill index**: for each skill, only its name and its
`when_to_use` paragraph.

**The point.** Selection reads the index. The full skill bodies are never shown to
the router. The cell measures both sizes.

**Harness computes.**

In [20]:
# --- chapter 3, verbatim -------------------------------------------------------


_FRONTMATTER = re.compile(r"\A---\n(.*?)\n---\n?", re.S)


_SECTION = re.compile(r"^<!--\s*section:\s*([\w.\-]+)\s*-->\s*$", re.M)


@dataclass(frozen=True)
class Artifact:
    kind: str                       # identity | project | persona | skill
    name: str
    version: str
    path: Path
    meta: Dict
    sections: Dict[str, str]        # id -> body, in file order
    digest: str                     # content hash of the file as authored

    @property
    def always_loaded(self) -> bool:
        return bool(self.meta.get("always_loaded"))

    @property
    def owns(self) -> List[str]:
        return list(self.meta.get("owns") or [])

    @property
    def axes(self) -> List[str]:
        return list(self.meta.get("axes") or [])


def parse(path: Path) -> Artifact:
    raw = path.read_text(encoding="utf-8")
    m = _FRONTMATTER.match(raw)
    if not m:
        raise ValueError(f"{path.name}: no frontmatter — not an artifact")
    meta = yaml.safe_load(m.group(1)) or {}
    body = raw[m.end():]

    # Split on the section markers. A body outside any marker is a file someone
    # wrote as prose; it would be unaddressable, so it is an error, not a warning.
    marks = list(_SECTION.finditer(body))
    if not marks:
        raise ValueError(f"{path.name}: no <!-- section: ... --> markers")
    if body[:marks[0].start()].strip():
        raise ValueError(f"{path.name}: text before the first section marker")
    sections = {}
    for i, mark in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(body)
        sections[mark.group(1)] = body[mark.end():end].strip()

    for missing in set(meta.get("owns") or []) - set(sections):
        raise ValueError(f"{path.name}: claims to own '{missing}' but has no such section")

    return Artifact(kind=meta["artifact"], name=meta["name"], version=str(meta["version"]),
                    path=path, meta=meta, sections=sections,
                    digest=hashlib.sha256(raw.encode()).hexdigest()[:12])


class SkillChoice(BaseModel):
    skill: str = Field(description="The name of the skill to load, exactly as listed.")
    why: str = Field(description="One short sentence.")


def catalogue(descriptions: Dict[str, str]) -> str:
    return "\n".join(f"- {name}: {desc.strip()}" for name, desc in descriptions.items())


def overlap(artifacts: Sequence[Artifact]) -> Dict[str, List[str]]:
    """Section ids claimed by more than one artifact."""
    claims: Dict[str, List[str]] = {}
    for a in artifacts:
        for section in a.owns:
            claims.setdefault(section, []).append(a.name)
    return {sid: names for sid, names in claims.items() if len(names) > 1}


def load_for_task(names: Sequence[str]) -> List[Artifact]:
    """Identity + project + the named artifacts, refusing an ambiguous surface."""
    chosen = ALWAYS + [ARTIFACTS[n] for n in names]
    clash = overlap(chosen)
    if clash:
        raise ValueError(
            "ambiguous surface — every section needs exactly one owner:\n" +
            "\n".join(f"   {sid}: claimed by {', '.join(owners)}"
                      for sid, owners in sorted(clash.items())))
    return chosen


_H3 = re.compile(r"^### (.+)$", re.M)


AXIS_LABELS = {"tone": "Tone", "audience": "Audience"}


def _heading_axes(heading: str) -> Dict[str, str]:
    """`Tone: casual, Audience: customer` -> {'tone': 'casual', 'audience': 'customer'}."""
    found = {}
    for part in heading.split(","):
        if ":" not in part:
            continue
        key, _, value = part.partition(":")
        key = key.strip().lower()
        if key in AXIS_LABELS:
            found[key] = value.strip().lower()
    return found


def slice_section(body: str, axes: Dict[str, str]) -> str:
    """Keep the preamble, and only the H3 blocks whose axes all match."""
    marks = list(_H3.finditer(body))
    if not marks:
        return body
    kept = [body[:marks[0].start()].strip()]
    for i, mark in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(body)
        heading, block = mark.group(1).strip(), body[mark.end():end].strip()
        wanted = _heading_axes(heading)
        # A heading with no axes at all is unconditional prose, so it stays.
        if wanted and any(axes.get(k) != v for k, v in wanted.items()):
            continue
        kept.append(block)
    return "\n\n".join(x for x in kept if x)


def _norm(line: str) -> str:
    return re.sub(r"\s+", " ", line.strip().lower())


#: Load order, and therefore precedence when the same paragraph appears twice.
#: Durable first: a rule the agent carries everywhere outranks a task-local
#: restatement of it, so the surviving copy is the one with the wider scope.
KIND_PRECEDENCE = ("identity", "project", "skill", "persona")


def in_load_order(artifacts: Sequence[Artifact]) -> List[Artifact]:
    return sorted(artifacts, key=lambda a: KIND_PRECEDENCE.index(a.kind))


@dataclass
class Surface:
    text: str
    manifest: Dict[str, object]

    @property
    def words(self) -> int:
        return len(self.text.split())


def compile_surface(*, artifacts: Sequence[Artifact], axes: Dict[str, str],
                    facts: Dict[str, object]) -> Surface:
    blocks, seen, dropped, used = [], {}, [], []
    for a in in_load_order(artifacts):
        for sid, body in a.sections.items():
            text = slice_section(body, axes) if a.axes else body
            # Deduplicate paragraph by paragraph, not section by section: the
            # same sentence written by two teams is the common case, two
            # identical sections is not. The copy that survives is the one from
            # the artifact earliest in load order, and the manifest records who
            # kept it — so "who says this?" has exactly one answer.
            kept = []
            for para in re.split(r"\n\s*\n", text):
                if not para.strip():
                    continue
                key = _norm(para)
                if key in seen:
                    dropped.append({"dropped_from": f"{a.name}:{sid}",
                                    "kept_from": seen[key],
                                    "text": para.strip()[:70]})
                    continue
                seen[key] = f"{a.name}:{sid}"
                kept.append(para.strip())
            if not kept:
                continue
            blocks.append("\n\n".join(kept))
            used.append({"artifact": a.name, "version": a.version,
                         "digest": a.digest, "section": sid})
    fact_lines = "\n".join(f"- {k}: {v}" for k, v in facts.items() if k != "changes")
    blocks.append(f"## Facts supplied by the harness\n\n{fact_lines}\n\n"
                  "These are given. Do not restate them as claims of your own, and "
                  "do not infer any fact that is not listed.")
    text = "\n\n".join(blocks)
    return Surface(text=text, manifest={
        "axes": dict(axes), "sections": used, "deduplicated": dropped,
        "surface_digest": hashlib.sha256(text.encode()).hexdigest()[:12]})


def everything_surface(artifacts: Sequence[Artifact], facts: Dict[str, object]) -> Surface:
    """Every artifact, every variant — the way a hand-maintained prompt grows."""
    blocks, used = [], []
    for a in artifacts:
        for sid, body in a.sections.items():
            blocks.append(body)
            used.append({"artifact": a.name, "version": a.version,
                         "digest": a.digest, "section": sid})
    fact_lines = "\n".join(f"- {k}: {v}" for k, v in facts.items() if k != "changes")
    blocks.append(f"## Facts supplied by the harness\n\n{fact_lines}")
    text = "\n\n".join(blocks)
    return Surface(text=text, manifest={"axes": {}, "sections": used, "deduplicated": [],
                                        "surface_digest": hashlib.sha256(text.encode()).hexdigest()[:12]})


class ReleaseNotes(BaseModel):
    summary: str = Field(description="One sentence, at most 25 words.")
    highlights: List[str] = Field(description="Three to six bullets, one change each.")


# --- wiring ---------------------------------------------------------------------
try:
    import tiktoken
    _enc = tiktoken.get_encoding("cl100k_base")
    def ntok(t): return len(_enc.encode(t))
except Exception:
    def ntok(t): return max(1, round(len(t) / 4))

ARTIFACTS = {a.name: a for a in (parse(p) for p in files)}
ALWAYS = [a for a in ARTIFACTS.values() if a.always_loaded]
LOADABLE = [a for a in ARTIFACTS.values() if not a.always_loaded]
AS_WRITTEN = {a.name: a.meta["when_to_use"] for a in LOADABLE if a.kind == "skill"}

print(f"{'artifact':22}{'kind':10}{'version':10}digest")
for a in ARTIFACTS.values():
    print(f"{a.name:22}{a.kind:10}{a.version:10}{a.digest}")

artifact              kind      version   digest
acme-docs-agent       identity  2.1.0     f13e99c92930
acme-data-platform    project   4.0.2     6246496911d7
plainspoken           persona   1.2.0     dc78ba66e414
incident-report       skill     1.1.0     a998ee689b8b
release-notes         skill     3.4.0     9a0b8bce844f


In [21]:
# The skill index: metadata only. This is all a router ever sees.
INDEX = {a.name: {"kind": a.kind, "version": a.version, "digest": a.digest, "axes": a.axes,
                  "when_to_use": a.meta.get("when_to_use", "").strip()}
         for a in LOADABLE if a.kind == "skill"}

print(catalogue(AS_WRITTEN), "\n")
for name, row in INDEX.items():
    print(f"   {name:<18} v{row['version']:<8} digest {row['digest']}   variables: {row['axes']}")

index_tokens = ntok(catalogue(AS_WRITTEN))
body_tokens = sum(ntok(a.path.read_text(encoding="utf-8")) for a in LOADABLE if a.kind == "skill")
print(f"\nindex handed to the router: {index_tokens} tokens; the two skill bodies it does NOT see: {body_tokens} tokens")

- incident-report: Use when writing the customer-facing report for an incident that has been resolved: what broke, who was affected, what was done. Not for release notes, and not while the incident is still open.
- release-notes: Use when turning a list of merged changes into the release notes for one version. Not for incidents, not for upgrade guides, not for a single changelog line. 

   incident-report    v1.1.0    digest a998ee689b8b   variables: ['audience']
   release-notes      v3.4.0    digest 9a0b8bce844f   variables: ['tone', 'audience']

index handed to the router: 81 tokens; the two skill bodies it does NOT see: 1216 tokens


## 3. Selection: one skill, one version

Two ways to pick the skill, and they must agree.

- **Model produces.** `route()` is Chapter 3's router: one typed call to the small
  `ROUTER` model at temperature 0. It receives the index and the task sentence, and
  must answer with a skill name from the list. `instructor` enforces the shape.
- **Harness computes.** The envelope already carries `task_type`. When a request
  names its task type, no model is needed; the lookup is the deterministic
  alternative.

The result of selection is not a name. It is a name **and a version and a digest**:
`ARTIFACTS[name]` is the record parsed in section 2.

In [22]:
def route(task: str, label: str) -> str:                     # chapter 3 §3, at temperature 0
    return typed_call(model=ROUTER, response_model=SkillChoice, label=label, temperature=0.0,
                      messages=[{"role": "system", "content":
                                 "Choose exactly one skill for the task. Answer with its name."},
                                {"role": "user", "content":
                                 f"SKILLS\n{catalogue(AS_WRITTEN)}\n\nTASK\n{task}"}]).skill


def route_by_envelope(req: Dict) -> str:
    """Deterministic alternative: the envelope names the task type, and it must be a known skill."""
    name = req.get("task_type")
    if name not in AS_WRITTEN:
        raise ValueError(f"task_type {name!r} is not a skill in the index: {sorted(AS_WRITTEN)}")
    return name


by_model = route(REQUEST["task"], label=f"route[demo1|{REQUEST['request_id']}]")          # model produces
by_envelope = route_by_envelope(REQUEST)                           # harness computes
print(f"router ({ROUTER}, temperature 0) chose : {by_model}")
print(f"envelope task_type says                 : {by_envelope}")
print(f"agree: {by_model == by_envelope}")

SKILL = ARTIFACTS[by_envelope]          # the harness-side answer is the one we bind to
print(f"\nselected: {SKILL.name} version {SKILL.version}, file digest {SKILL.digest}, "
      f"owns {len(SKILL.owns)} sections, variables {SKILL.axes}")

router (openai/gpt-5.4-mini, temperature 0) chose : release-notes
envelope task_type says                 : release-notes
agree: True

selected: release-notes version 3.4.0, file digest 9a0b8bce844f, owns 7 sections, variables ['tone', 'audience']


## 4. Resolution: the task variables come from the envelope

Each skill declares which **axes** (variables) its rules vary on:
`release-notes` has `tone` and `audience`; `incident-report` has only `audience`.
The harness reads those values from the envelope and checks that the skill actually
has a variant for them, by scanning the skill's own `### Tone: ..., Audience: ...`
headings.

**The point.** An envelope value the skill does not know is refused here, before
compilation, with a message that names the known values.

**Harness computes.**

In [23]:
def known_axis_values(skill: Artifact) -> Dict[str, set]:
    """The axis values a skill actually has rules for, read from its own ### headings."""
    known: Dict[str, set] = {}
    for body in skill.sections.values():
        for m in _H3.finditer(body):
            for k, v in _heading_axes(m.group(1)).items():
                known.setdefault(k, set()).add(v)
    return known


def resolve_axes(req: Dict, skill: Artifact) -> Dict[str, str]:
    """Only the axes this skill declares, with values the skill has a variant for."""
    known = known_axis_values(skill)
    axes = {}
    for axis in skill.axes:
        if axis not in req:
            raise ValueError(f"the envelope has no '{axis}', which {skill.name} needs")
        value = str(req[axis]).strip().lower()
        if value not in known.get(axis, {value}):
            raise ValueError(f"{skill.name} has no variant for {axis}={value!r}; it knows {sorted(known[axis])}")
        axes[axis] = value
    return axes


AXES = resolve_axes(REQUEST, SKILL)
print(f"{SKILL.name:<16} declares {SKILL.axes} -> resolved {AXES}")
print(f"{'incident-report':<16} declares {ARTIFACTS['incident-report'].axes} -> resolved "
      f"{resolve_axes(REQUEST, ARTIFACTS['incident-report'])}   (tone is ignored: that skill has no tone rules)")

try:
    resolve_axes({**REQUEST, "tone": "playful"}, SKILL)
except ValueError as exc:
    print(f"\nrefused: {exc}")

release-notes    declares ['tone', 'audience'] -> resolved {'tone': 'casual', 'audience': 'customer'}
incident-report  declares ['audience'] -> resolved {'audience': 'customer'}   (tone is ignored: that skill has no tone rules)

refused: release-notes has no variant for tone='playful'; it knows ['casual', 'formal']


## 5. Compilation: only the applicable instructions, in one surface

`load_for_task` picks identity + project + the selected skill + the persona.
`compile_surface` then, for every section:

- keeps the preamble and only the `###` blocks whose axes match (`slice_section`);
- drops a paragraph that an earlier artifact already said (deduplication), and
  records who kept it;
- appends the **facts supplied by the harness** (version, breaking flag).

The output is a `Surface`: the text, and a manifest listing every section used with
its artifact version and digest, and a `surface_digest` of the text.

**The point.** The manifest answers "who says this?" with exactly one artifact per
paragraph.

**Harness computes.**

In [24]:
FACTS = {"version": REQUEST["payload"]["version"], "breaking": REQUEST["payload"]["breaking"]}

LOADED = load_for_task([SKILL.name, "plainspoken"])            # identity + project + skill + persona
SURFACE = compile_surface(artifacts=LOADED, axes=AXES, facts=FACTS)

print(f"surface: {ntok(SURFACE.text)} tokens, {SURFACE.words} words, digest {SURFACE.manifest['surface_digest']}")
print(f"axes: {SURFACE.manifest['axes']}\n")
print(f"{'artifact':<20}{'version':<9}{'digest':<14}section")
for s in SURFACE.manifest["sections"]:
    print(f"{s['artifact']:<20}{s['version']:<9}{s['digest']:<14}{s['section']}")
print("\ndeduplicated paragraphs:")
for d in SURFACE.manifest["deduplicated"]:
    print(f"   dropped from {d['dropped_from']:<28} kept from {d['kept_from']:<28} \"{d['text']}...\"")

surface: 788 tokens, 584 words, digest 61368dc1fa99
axes: {'tone': 'casual', 'audience': 'customer'}

artifact            version  digest        section
acme-docs-agent     2.1.0    f13e99c92930  identity.role
acme-docs-agent     2.1.0    f13e99c92930  identity.refusals
acme-docs-agent     2.1.0    f13e99c92930  identity.escalation
acme-data-platform  4.0.2    6246496911d7  project.product
acme-data-platform  4.0.2    6246496911d7  project.versioning
acme-data-platform  4.0.2    6246496911d7  project.audiences
release-notes       3.4.0    9a0b8bce844f  output.contract
release-notes       3.4.0    9a0b8bce844f  summary.style
release-notes       3.4.0    9a0b8bce844f  highlights.structure
release-notes       3.4.0    9a0b8bce844f  address.contract
release-notes       3.4.0    9a0b8bce844f  prohibitions.hard
release-notes       3.4.0    9a0b8bce844f  typography
release-notes       3.4.0    9a0b8bce844f  examples
plainspoken         1.2.0    dc78ba66e414  persona.voice

deduplicated paragr

### Flip the audience and watch the surface change

Same request, one field changed: `audience` goes from `customer` to
`developer`. Nothing else moves. The cell prints the `summary.style` section as
authored, then as compiled for each audience, side by side, with line and token
counts, and the two whole-surface digests.

**The point.** Compilation is a function of the envelope. One variable, two different
rule sets, two different digests, no model involved.

**Harness computes.**

In [25]:
REQUEST_DEV = {**REQUEST, "audience": "developer"}            # the flip: one field
AXES_DEV = resolve_axes(REQUEST_DEV, SKILL)

body = SKILL.sections["summary.style"]
cust, dev = slice_section(body, AXES), slice_section(body, AXES_DEV)
print(f"summary.style as authored      : {len(body.splitlines()):>2} lines, {ntok(body):>3} tokens (every variant)")
print(f"compiled for audience=customer : {len(cust.splitlines()):>2} lines, {ntok(cust):>3} tokens")
print(f"compiled for audience=developer: {len(dev.splitlines()):>2} lines, {ntok(dev):>3} tokens\n")

W = 58
left = [l for p in cust.split("\n\n") for l in textwrap.wrap(p, W) + [""]]
right = [l for p in dev.split("\n\n") for l in textwrap.wrap(p, W) + [""]]
print(f"{'audience=customer':<{W}}   |   audience=developer")
print("-" * W + "   |   " + "-" * W)
for a, b in zip(left + [""] * (len(right) - len(left)), right + [""] * (len(left) - len(right))):
    print(f"{a:<{W}}   |   {b}")

S_DEV = compile_surface(artifacts=LOADED, axes=AXES_DEV, facts=FACTS)
print(f"\nwhole surface, customer : {ntok(SURFACE.text):>4} tokens, digest {SURFACE.manifest['surface_digest']}")
print(f"whole surface, developer: {ntok(S_DEV.text):>4} tokens, digest {S_DEV.manifest['surface_digest']}")

summary.style as authored      : 13 lines,  96 tokens (every variant)
compiled for audience=customer :  6 lines,  57 tokens
compiled for audience=developer:  6 lines,  55 tokens

audience=customer                                            |   audience=developer
----------------------------------------------------------   |   ----------------------------------------------------------
## 2. Summary                                                |   ## 2. Summary
                                                             |   
One sentence, at most 25 words. It names what changed, not   |   One sentence, at most 25 words. It names what changed, not
why it is good.                                              |   why it is good.
                                                             |   
Write it for an operator using the web console. Name the     |   Write it for someone integrating against the API. Name the
feature, not the endpoint. Do not mention status codes,      |   affected

### Measured against "inject everything"

`everything_surface` is the alternative most teams start with: every
artifact, every variant, in one hand-maintained prompt. Same facts appended. The
cell measures tokens for both, and for all four axis combinations of this skill.

**Harness computes.**

In [26]:
EVERYTHING = everything_surface(list(ARTIFACTS.values()), FACTS)

print(f"compiled surface   : {ntok(SURFACE.text):>6} tokens   digest {SURFACE.manifest['surface_digest']}")
print(f"everything surface : {ntok(EVERYTHING.text):>6} tokens   digest {EVERYTHING.manifest['surface_digest']}")
print(f"saved              : {1 - ntok(SURFACE.text) / ntok(EVERYTHING.text):>6.0%}\n")

print(f"{'tone':<10}{'audience':<12}{'tokens':>8}  surface digest")
for tone in ("casual", "formal"):
    for audience in ("customer", "developer"):
        s = compile_surface(artifacts=LOADED, axes={"tone": tone, "audience": audience}, facts=FACTS)
        print(f"{tone:<10}{audience:<12}{ntok(s.text):>8}  {s.manifest['surface_digest']}")
print("\nfour requests, four different surfaces, each with its own digest; the everything surface is one text for all of them")

compiled surface   :    788 tokens   digest 61368dc1fa99
everything surface :   1382 tokens   digest f9d27d9f7b46
saved              :    43%

tone      audience      tokens  surface digest
casual    customer         788  61368dc1fa99
casual    developer        829  d43c2bf1becd
formal    customer         796  bafe0a0338aa
formal    developer        860  8e303b946179

four requests, four different surfaces, each with its own digest; the everything surface is one text for all of them


## 6. What the build refuses

Three refusals, all before any model call:

1. **Ambiguous owner.** Both skills own a section called `output.contract`. Loading
   both at once would give one rule two authors, so `load_for_task` refuses.
2. **Malformed artifact.** `parse` refuses a file with prose outside any section
   marker, or one that claims to own a section it does not contain.
3. **Deprecated version.** The Drive artifacts carry no deprecation flag, so the
   cell writes a copy of `release-notes` marked `status: deprecated` into a side
   folder and shows a small resolver that refuses it. This is the one piece of
   code in this notebook that is not from Chapter 3.

**Harness computes.**

In [27]:
# 1. ambiguous owner
try:
    load_for_task(["release-notes", "incident-report"])
except ValueError as exc:
    print(f"refused (overlap):\n{textwrap.indent(str(exc), '   ')}\n")

# 2. malformed artifacts
BAD = WORK / "bad"
BAD.mkdir(exist_ok=True)
(BAD / "prose-first.md").write_text("---\nartifact: skill\nname: x\nversion: 1.0.0\nowns: [a]\n---\n"
                                    "Some prose nobody can address.\n<!-- section: a -->\nbody\n", encoding="utf-8")
(BAD / "claims-missing.md").write_text("---\nartifact: skill\nname: y\nversion: 1.0.0\nowns: [a, b]\n---\n"
                                       "<!-- section: a -->\nbody\n", encoding="utf-8")
for p in sorted(BAD.glob("*.md")):
    try:
        parse(p)
    except ValueError as exc:
        print(f"refused (parse): {exc}")

# 3. deprecated version (not in the Drive artifacts; a local copy marked deprecated)
VERSIONS_DIR = WORK / "versions"
VERSIONS_DIR.mkdir(exist_ok=True)
stale = SKILL.path.read_text(encoding="utf-8").replace(
    f"version: {SKILL.version}", f"version: 3.3.0\nstatus: deprecated\nsuperseded_by: {SKILL.version}", 1)
(VERSIONS_DIR / "skill-release-notes-3.3.0.md").write_text(stale, encoding="utf-8")
VERSIONS = {(a.name, a.version): a for a in [*ARTIFACTS.values(), parse(VERSIONS_DIR / "skill-release-notes-3.3.0.md")]}


def resolve(name: str, version: Optional[str] = None) -> Artifact:
    """The newest non-deprecated version, or exactly the version asked for if it is still allowed."""
    found = [a for (n, v), a in VERSIONS.items() if n == name and (version is None or v == version)]
    if not found:
        raise LookupError(f"no {name} {version or ''}".strip())
    a = max(found, key=lambda a: tuple(int(x) for x in a.version.split(".")))
    if a.meta.get("status") == "deprecated":
        raise ValueError(f"{name} {a.version} is deprecated, superseded by {a.meta.get('superseded_by')}")
    return a


print(f"\nresolve('release-notes')          -> {resolve('release-notes').version} ({resolve('release-notes').digest})")
try:
    resolve("release-notes", "3.3.0")
except ValueError as exc:
    print(f"resolve('release-notes', '3.3.0') -> refused: {exc}")

refused (overlap):
   ambiguous surface — every section needs exactly one owner:
      output.contract: claimed by release-notes, incident-report
      prohibitions.hard: claimed by release-notes, incident-report
      summary.style: claimed by release-notes, incident-report
      typography: claimed by release-notes, incident-report

refused (parse): claims-missing.md: claims to own 'b' but has no such section
refused (parse): prose-first.md: text before the first section marker

resolve('release-notes')          -> 3.4.0 (9a0b8bce844f)
resolve('release-notes', '3.3.0') -> refused: release-notes 3.3.0 is deprecated, superseded by 3.4.0


## 7. The execution, and the record of it

Now, and only now, the writer runs: one typed call to `WRITER` at
temperature 0, system prompt = the compiled surface, user message = the facts from
the envelope. The response is forced into the `ReleaseNotes` shape (summary +
highlights) by `instructor`.

Then the harness writes the **record**: which skill at which version and digest,
which artifacts went into the surface, the surface digest, the resolved variables,
the model binding and the measured usage. Three weeks from now this dict is how you
answer "what exactly did this run read?".

Below, **model produces** is marked; everything else in the cell is **harness computes**.

In [28]:
USER_MESSAGE = (f"Release {FACTS['version']} of the Acme data platform.\n"
                + "\n".join(f"- {c}" for c in REQUEST["payload"]["changes"]))

# --- model produces -------------------------------------------------------------
notes = typed_call(model=WRITER, response_model=ReleaseNotes, temperature=0.0, label=f"write[demo1|{REQUEST['request_id']}]",
                   messages=[{"role": "system", "content": SURFACE.text},
                             {"role": "user", "content": USER_MESSAGE}])
print("MODEL PRODUCED")
print(f"   summary  : {notes.summary}")
for h in notes.highlights:
    print(f"   highlight: {h}")

# --- harness computes: the record ------------------------------------------------
TRACE = {
    "request_id": REQUEST["request_id"],
    "selection": {"skill": SKILL.name, "version": SKILL.version, "digest": SKILL.digest,
                  "by_envelope": by_envelope, "by_router": {"model": ROUTER, "temperature": 0.0, "choice": by_model}},
    "variables": AXES,
    "facts": FACTS,
    "artifacts": [{"name": a.name, "version": a.version, "digest": a.digest} for a in in_load_order(LOADED)],
    "surface": {"digest": SURFACE.manifest["surface_digest"], "tokens": ntok(SURFACE.text),
                "sections": [f"{s['artifact']}:{s['section']}" for s in SURFACE.manifest["sections"]]},
    "writer": {"model": WRITER, "temperature": 0.0, "mode": "instructor.Mode.TOOLS", "response_model": "ReleaseNotes"},
    "usage": {k: USAGE[-1][k] for k in ("prompt_tokens", "completion_tokens")},
    "output_digest": hashlib.sha256(notes.model_dump_json().encode()).hexdigest()[:12],
}
print("\nHARNESS RECORDED")
print(textwrap.indent(json.dumps(TRACE, indent=2), "   "))

MODEL PRODUCED
   summary  : We rebuilt the CSV exporter, improving speed, memory use, CSV quoting, and export job handling.
   highlight: Export about twice as fast
   highlight: Use roughly 40 percent less peak memory during export
   highlight: Escape quotes correctly inside CSV fields
   highlight: Receive a 202 response and job id from `/v1/export`
   highlight: Poll `/v1/export/{id}` to retrieve finished exports

HARNESS RECORDED
   {
     "request_id": "req-0917-001",
     "selection": {
       "skill": "release-notes",
       "version": "3.4.0",
       "digest": "9a0b8bce844f",
       "by_envelope": "release-notes",
       "by_router": {
         "model": "openai/gpt-5.4-mini",
         "temperature": 0.0,
         "choice": "release-notes"
       }
     },
     "variables": {
       "tone": "casual",
       "audience": "customer"
     },
     "facts": {
       "version": "2.4.0",
       "breaking": true
     },
     "artifacts": [
       {
         "name": "acme-docs-agent",
 

## 8. Two one-character edits, one declared version

Copy the skill file into a side folder and make two separate edits, each
leaving `version: 3.4.0` exactly as it is:

1. in a rule this request reads: `at most 25 words` becomes `at most 24 words`;
2. in a block this request does **not** read: the `Tone: formal` example
   `Improved export throughput` gets an `s`.

Parse and compile again with the same variables. The cell prints one table with three
columns: declared version, artifact digest, surface digest.

**The point.** Both edits change the artifact digest. Only the applicable one changes
the surface digest. The declared version changes in neither case. That is why
recording the version alone cannot tell you what the writer saw; the record needs
the artifact digest (did the file change?) and the surface digest (did what this
request read change?).

**Harness computes.**

In [29]:
EDITED = WORK / "edited"
EDITED.mkdir(exist_ok=True)
original = SKILL.path.read_text(encoding="utf-8")


def recompiled(edited_text: str, filename: str):
    p = EDITED / filename
    p.write_text(edited_text, encoding="utf-8")
    a = parse(p)
    s = compile_surface(artifacts=ALWAYS + [a, ARTIFACTS["plainspoken"]], axes=AXES, facts=FACTS)
    return a, s


edits = {
    "as shipped": original,
    "'25 words' -> '24 words' (a rule this request reads)": original.replace("at most 25 words", "at most 24 words", 1),
    "'throughput' -> 'throughputs' in Tone: formal (not read here)": original.replace(
        "`Improved export throughput`.", "`Improved export throughputs`.", 1),
}
assert all(t != original for k, t in edits.items() if k != "as shipped"), "an edit did not apply"

print(f"{'edit':<62}{'declared version':<18}{'artifact digest':<17}surface digest")
for label, text in edits.items():
    a, s = recompiled(text, f"{label[:12].replace(' ', '_').replace(chr(39), '')}.md")
    print(f"{label:<62}{a.version:<18}{a.digest:<17}{s.manifest['surface_digest']}")

edit                                                          declared version  artifact digest  surface digest
as shipped                                                    3.4.0             9a0b8bce844f     61368dc1fa99
'25 words' -> '24 words' (a rule this request reads)          3.4.0             9c46fdc053a9     3d91d63ad111
'throughput' -> 'throughputs' in Tone: formal (not read here) 3.4.0             5a7122bc0640     61368dc1fa99


## 9. What this run cost

Every call made above is in `USAGE`; prices come from OpenRouter's published
per-token rates.

In [30]:
@lru_cache(maxsize=1)
def pricing() -> Dict[str, Tuple[float, float]]:
    req = urllib.request.Request(f"{BASE_URL}/models",
                                 headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"})
    data = json.load(urllib.request.urlopen(req, timeout=60))["data"]
    return {m["id"]: (float(m["pricing"]["prompt"]), float(m["pricing"]["completion"])) for m in data}


def cost_of(u) -> float:
    p_in, p_out = pricing().get(u["model"], (0.0, 0.0))
    return u["prompt_tokens"] * p_in + u["completion_tokens"] * p_out


agg: Dict[Tuple[str, str], Dict[str, float]] = {}
for u in USAGE:
    role, _, rest = u["label"].partition("[")
    a = agg.setdefault((role, rest.split("|")[0]), {"n": 0, "in": 0, "out": 0, "usd": 0.0})
    a["n"] += 1; a["in"] += u["prompt_tokens"]; a["out"] += u["completion_tokens"]; a["usd"] += cost_of(u)

print(f"{'role':<8}{'harness':<14}{'calls':>7}{'tok in':>12}{'tok out':>10}{'USD':>10}")
for (role, who), a in sorted(agg.items()):
    print(f"{role:<8}{who:<14}{a['n']:>7}{a['in']:>12,}{a['out']:>10,}{a['usd']:>10.5f}")
print(f"{'total':<22}{sum(a['n'] for a in agg.values()):>7}{sum(a['in'] for a in agg.values()):>12,}"
      f"{sum(a['out'] for a in agg.values()):>10,}{sum(a['usd'] for a in agg.values()):>10.5f}")
print("\nPriced from the published per-token rates against measured tokens; treat it as the\n"
      "cost of the work, not a reconciliation of an account.")

role    harness         calls      tok in   tok out       USD
route   demo1               1         201        36   0.00031
write   demo1               1         953       201   0.00043
total                       2       1,154       237   0.00074

Priced from the published per-token rates against measured tokens; treat it as the
cost of the work, not a reconciliation of an account.


## What we just built

| step | harness or model | what it did | what it recorded |
|---|---|---|---|
| **discovery** | harness | parsed five files into typed records; built an index of names, versions, descriptions | kind, version, digest per artifact |
| **selection** | model (router, temperature 0) and a deterministic lookup | picked one skill; bound to one version | skill name, version, digest, which path chose it |
| **resolution** | harness | read `audience` and `tone` from the envelope; refused values the skill has no variant for | the axes |
| **compilation** | harness | sliced sections to the axes, deduplicated paragraphs, appended facts | manifest: sections used, who kept each paragraph, surface digest |
| **refusals** | harness | ambiguous owner, malformed file, deprecated version | an error before any model call |
| **the record** | harness around one model call | ran the writer once on the compiled surface | everything above plus model binding, usage, output digest |

The measured gap between the compiled surface and the everything surface is printed
in section 5; the digests in section 8 show that a one-character edit to a rule is
visible in the record while the declared version is not.

**Takeaway: settle everything the harness can decide before calling a model.** The
skill, its version, the variables, the applicable rules and the facts were all
fixed, checked and written down before the writer saw a single token. The model was
left with the one job only a model can do: write the sentences.

*Continues in Demo 2:* the same desk over a whole shift of events, where the facts
arrive with corrections, and the harness has to carry state between steps.

## Handoff to hands-on 1: what schema validation buys, and what it does not

Look again at the release notes the writer produced in section 7. The
response came back in the shape the harness demanded: `instructor` validated it
against `ReleaseNotes` (a `summary` string, a `highlights` list of strings), so the
structure is guaranteed. The cell then runs two checks the schema cannot express: a
customer-facing rule from the skill ("name the feature, not the endpoint") and the
identity's refusal to state a number that is not in the input.

**Harness computes**; the notes are the saved **model output** from section 7, unchanged.

In [31]:
print("saved release notes from section 7 (the writer's output, unchanged):")
print(f"   summary  : {notes.summary}")
for h in notes.highlights:
    print(f"   highlight: {h}")

# 1. structure: what the schema guarantees
ReleaseNotes.model_validate(notes.model_dump())
print(f"\nschema validation : passed  (summary: str; highlights: list[str] with {len(notes.highlights)} items)")

# 2. meaning: what the schema cannot see
lines = [notes.summary] + notes.highlights
api_leaks = [l for l in lines if re.search(r"/v1/|\b202\b|job id", l)]
print(f"customer rule 'name the feature, not the endpoint': "
      f"{'violated in ' + str(len(api_leaks)) + ' line(s)' if api_leaks else 'respected'}")
for l in api_leaks:
    print(f"   -> {l}")
NUM = r"\d+(?:[.:]\d+)*"
input_text = USER_MESSAGE + " " + json.dumps(FACTS)
unsourced = sorted(set(re.findall(NUM, " ".join(lines))) - set(re.findall(NUM, input_text)))
print(f"numbers in the output that are not in the input: {unsourced or 'none'}")
print("\nthe schema said yes to all of this; whether it may ship is a different question, and a different check")

saved release notes from section 7 (the writer's output, unchanged):
   summary  : We rebuilt the CSV exporter, improving speed, memory use, CSV quoting, and export job handling.
   highlight: Export about twice as fast
   highlight: Use roughly 40 percent less peak memory during export
   highlight: Escape quotes correctly inside CSV fields
   highlight: Receive a 202 response and job id from `/v1/export`
   highlight: Poll `/v1/export/{id}` to retrieve finished exports

schema validation : passed  (summary: str; highlights: list[str] with 5 items)
customer rule 'name the feature, not the endpoint': violated in 2 line(s)
   -> Receive a 202 response and job id from `/v1/export`
   -> Poll `/v1/export/{id}` to retrieve finished exports
numbers in the output that are not in the input: none

the schema said yes to all of this; whether it may ship is a different question, and a different check


**Remember.** Schema validation guarantees shape, not compliance. The
checks above are the first kind of evaluation beyond the schema: deterministic rules written against the compiled surface.